# Your slides stay with you

**Prepare an H&E slide on your own machine. No image is uploaded.**

DeepSpot-H, the foundation model for H&E images, turns each tile of your slide
into one embedding. This notebook runs it here, writes those embeddings to one
file, and opens that file so you can check what it holds.

1. Fetch an example slide
2. Open the slide
3. Ask the service what to use
4. Check the tiles
5. Run DeepSpot-H
6. Check what the file holds
7. Explore the embeddings

The file is what DeepSpot-M, the model that predicts spatial gene expression,
reads. [deepspot-m-quickstart][m] continues from it.

[m]: https://github.com/auroraomics/deepspot-m-quickstart

## Before you start

From the repository root:

```bash
pip install -r requirements.txt
auroraomics login you@institution.edu
```

Steps 3 and 5 ask the service, so they need your key. [Get access][access] if
you do not have one. Access is granted per account, so ask before you need it.

[access]: https://docs.auroraomics.org/get-access/


---
## 0. Check the install

If the import fails, install the requirements in this kernel's environment.
Step 5 is the first that needs the `embed` extra.


In [ ]:
import subprocess
import sys
from importlib import metadata
from pathlib import Path

assert sys.version_info >= (3, 10), f"Python 3.10 or newer is needed; this is {sys.version.split()[0]}"

# The folder this notebook lives in, whichever directory the kernel started in.
HERE = Path.cwd() if (Path.cwd() / "example_slide.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(HERE))

import matplotlib.pyplot as plt
import numpy as np

import auroraomics as ao
from auroraomics import contracts, pack

print("python      ", sys.version.split()[0])
print("auroraomics ", metadata.version("auroraomics"))
print("numpy       ", np.__version__)


---
## 1. Fetch an example slide

One lung cancer section, `LC1`, from an open-access dataset:

> Dawo, S., Nonchev, K., & Silina, K. (2025). *10x Visium Spatial
> Transcriptomics Dataset: Kidney (3) and Lung (5) Cancer with Tertiary
> Lymphoid Structures* [Data set]. Zenodo.
> <https://doi.org/10.5281/zenodo.14620362>

The cell below prints the citation each time it runs. `example_slide.py`
fetches this one slide without downloading the whole archive, and verifies it
before writing it.

To use your own slide, point `SLIDE` at it and delete the last line of this
cell.


In [ ]:
from example_slide import CITATION, SLIDE_MEMBER, fetch_member

DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
SLIDE = DATA / "LC1.tif"

print("Example data:", CITATION)
print()

fetch_member(SLIDE_MEMBER, SLIDE)


---
## 2. Open the slide

`open_slide` reads the slide's resolution from the file, and refuses a file
that records none rather than assuming one. If your slide records none, pass it
with `ao.open_slide(path, mpp=...)`.

Only the part being read is decoded, so a slide larger than memory works.


In [ ]:
slide = ao.open_slide(SLIDE)
height, width = slide.shape[:2]

print(slide)
print(f"{width:,} x {height:,} px at {slide.mpp:.4f} um/px")
print(f"which is {width * slide.mpp / 1000:.1f} x {height * slide.mpp / 1000:.1f} mm of tissue")


In [ ]:
def overview(slide, max_edge=900):
    """A small picture of the whole slide, read one band of rows at a time."""
    height, width = slide.shape[:2]
    step = max(1, -(-max(height, width) // max_edge))
    band = step * 128
    return np.concatenate(
        [
            np.asarray(slide[y : min(y + band, height), 0:width])[::step, ::step]
            for y in range(0, height, band)
        ],
        axis=0,
    )


small = overview(slide)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(small)
ax.set_title(f"The section: {width:,} x {height:,} px at {slide.mpp:.3f} um/px", fontsize=10)
ax.set_axis_off()
plt.show()


---
## 3. Ask the service what to use

Three answers come from the service: the model that reads embeddings, with the
tile size its card states; the release of DeepSpot-H to run; and the quality
standard every tile is held to. Your machine prepares the slide the way the
service would.

None of these calls sends anything about your slide.


In [ ]:
with ao.Client() as client:
    cards = client.models()
    registry = client.encoders()
    thresholds = client.qc_thresholds()

card = next(c for c in cards if "embeddings" in c["accepts"]["observations"])
model_id = card["id"]
patch_um = float(card["input_spec"]["patch_um"])   # the tile size the card states
encoder = ao.resolve_encoder(registry)             # DeepSpot-H, as the service pins it

print("model           ", model_id)
print("quality standard", thresholds.as_dict())


---
## 4. Check the tiles

The slide is cut at the tile size the model's card states, and every tile is
judged before DeepSpot-H runs: enough tissue, in focus, stained. The check runs
here, and the maps below show which tiles are kept.


In [ ]:
edge = pack.tile_edge_px(patch_um, slide.mpp)
cols, rows = width // edge, height // edge

candidates = [
    (int((i + 0.5) * edge), int((j + 0.5) * edge))
    for j in range(rows)
    for i in range(cols)
]

verdicts = np.array(
    [ao.qc_tile(pack.tile_at(slide, cx, cy, edge), thresholds).passes for cx, cy in candidates]
)
positions = np.array(candidates)
wanted = [c for c, ok in zip(candidates, verdicts) if ok]

print(f"{len(candidates):,} candidate tiles")
print(f"{len(wanted):,} meet the quality standard, {int((~verdicts).sum()):,} are dropped")
print(f"one submission carries at most {contracts.max_items('embeddings'):,} tiles")


In [ ]:
def on_the_grid(x, y, step, values, background=1.0):
    """Lay one value per tile out on the grid the tiles were cut on."""
    col = np.rint((x - x.min()) / step).astype(int)
    row = np.rint((y - y.min()) / step).astype(int)
    canvas = np.full((row.max() + 1, col.max() + 1) + np.shape(values)[1:], background, dtype=float)
    canvas[row, col] = values
    return canvas


kept_map = on_the_grid(
    positions[:, 0],
    positions[:, 1],
    edge,
    np.where(verdicts[:, None], [0.12, 0.37, 0.75], [0.93, 0.93, 0.93]),
)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.4))
axes[0].imshow(small)
axes[0].set_title("The section", fontsize=10)
axes[1].imshow(kept_map, interpolation="nearest")
axes[1].set_title(
    f"One pixel per candidate tile: {int(verdicts.sum()):,} kept, {int((~verdicts).sum()):,} dropped",
    fontsize=10,
)
for ax in axes:
    ax.set_axis_off()
plt.show()


In [ ]:
rng = np.random.default_rng(0)
picks = rng.choice(np.flatnonzero(verdicts), size=6, replace=False)

fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for ax, i in zip(axes, picks):
    cx, cy = candidates[i]
    ax.imshow(pack.tile_at(slide, cx, cy, edge))
    ax.set_title(f"({cx}, {cy})", fontsize=8)
    ax.set_axis_off()
fig.suptitle("Six kept tiles", fontsize=10)
plt.show()


---
## 5. Run DeepSpot-H

### First, check this machine

`doctor` runs DeepSpot-H once on a reference image shipped with the package,
and confirms this machine computes what the model reads. `--tiles` is how many
tiles you mean to embed.
`--allow-download` fetches DeepSpot-H's weights once, if this machine does not
hold them yet.

It runs on a CPU. A GPU is used when there is one.


In [ ]:
subprocess.run(
    ["auroraomics", "doctor", "--model", model_id, "--tiles", str(len(wanted)), "--allow-download"],
    check=False,
)


### Embed the tiles

`embed_tiles` reads, checks and embeds the tiles one batch at a time, so the
slide never has to fit in memory. `mpp` comes from the slide file, `patch_um`
from the model's card, and DeepSpot-H's release and the quality standard from
the service.

To try a corner of the slide first, set `REGION` to a window in full-resolution
pixels.


In [ ]:
REGION = None                # or (x0, y0, x1, y1) in full-resolution pixels
OUT = DATA / "LC1-embeddings.npz"

todo = wanted
if REGION is not None:
    x0, y0, x1, y1 = REGION
    todo = [(cx, cy) for cx, cy in wanted if x0 <= cx < x1 and y0 <= cy < y1]

print(f"embedding {len(todo):,} tiles into {OUT.name}")

report = ao.embed_tiles(
    slide,
    OUT,
    mpp=slide.mpp,             # read from the slide file
    patch_um=patch_um,         # the tile size the model's card states
    encoder=encoder,           # DeepSpot-H, as the service pins it
    thresholds=thresholds,     # the service's quality standard
    centers=todo,
)

print(f"{report.rows:,} of {report.considered:,} tiles embedded")
print("dropped by:", report.rejected_by_reason or "nothing")


---
## 6. Check what the file holds

For each tile, the file holds its embedding, its position on the slide and the
quality measurements that kept it, and it records how it was made. There is no
tile, no thumbnail, no slide label and no macro image in it.

The cell below lists the file's members and compares them with the contract the
package ships for this file.


In [ ]:
written = np.load(OUT, allow_pickle=False)
contract = contracts.embeddings_npz()

print(f"{OUT.name}: {OUT.stat().st_size / 1e6:.1f} MB, {len(written['emb']):,} tiles")
print("members:", ", ".join(sorted(written.files)))

missing = [name for name in contract["required"] if name not in written.files]
extra = [name for name in written.files if name not in contract["keys"] + contract["attrs"]]
print("required by the contract and missing:", missing or "none")
print("present and not named by the contract:", extra or "none")


---
## 7. Explore the embeddings

Two pictures show what the embeddings hold.

**A map.** Each tile's embedding becomes a colour, laid back where the tile came
from.

**Neighbours.** One tile, and the five whose embeddings are closest to it, to
compare by eye.


In [ ]:
emb = written["emb"].astype(np.float32)
x, y, step = written["px_col"], written["px_row"], int(written["patch_px"])

centred = emb - emb.mean(axis=0)
_, _, axes_of_variation = np.linalg.svd(centred, full_matrices=False)
scores = centred @ axes_of_variation[:3].T

low, high = np.percentile(scores, [2, 98], axis=0)
colour = np.clip((scores - low) / np.maximum(high - low, 1e-9), 0, 1)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(on_the_grid(x, y, step, colour), interpolation="nearest")
ax.set_title(f"{len(emb):,} tiles, coloured by their embeddings", fontsize=10)
ax.set_axis_off()
plt.show()


In [ ]:
unit = emb / np.linalg.norm(emb, axis=1, keepdims=True)
query = int(np.argmax(scores[:, 0]))
closest = np.argsort(-(unit @ unit[query]))[:6]

fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for rank, (ax, i) in enumerate(zip(axes, closest)):
    ax.imshow(pack.tile_at(slide, int(x[i]), int(y[i]), step))
    ax.set_title("Query" if rank == 0 else f"#{rank}", fontsize=9)
    ax.set_axis_off()
fig.suptitle("One tile, and the five whose embeddings are closest to it", fontsize=10)
plt.show()

slide.close()


---
## 8. Next

The slide, and every tile cut from it, stayed on this machine.

**Predict spatial gene expression.** [deepspot-m-quickstart][m] continues from
the file you wrote: DeepSpot-M turns these embeddings into spatial gene
expression on our service, and no image is sent.

**Keep the embeddings.** The file is a result on its own: one row per tile, with
the position each row came from.

[m]: https://github.com/auroraomics/deepspot-m-quickstart

| | |
|---|---|
| [DeepSpot-H](https://docs.auroraomics.org/models/deepspot-h/) | The foundation model for H&E images |
| [Your slides stay with you](https://docs.auroraomics.org/guides/embeddings/) | This route in the docs |
| [Check tile quality](https://docs.auroraomics.org/check-tiles/) | Why a tile was dropped |
| [Limits](https://docs.auroraomics.org/limits/) | How much one submission carries |
| [Your data, your workflow](https://docs.auroraomics.org/privacy/) | Where your data goes |
